# Hit callers on well profiles

{func}`~mantispy.tl.hit_calling` scores a treatment against the negative controls. On well profiles, where a
treatment has only its replicate wells, two of its methods apply, and this page runs both on two compound screens:

- `"mahalanobis"`, the default: the median distance of the treatment's wells from the control centroid, in the
  controls' covariance.
- `"mp_value"`: the squared Mahalanobis distance between the treatment's centroid and the controls', in the average
  of the two covariances. This is the multidimensional perturbation value (Hutz et al., *J Biomol Screen* 2013,
  [doi:10.1177/1087057112469257](https://doi.org/10.1177/1087057112469257)), and unlike the default it counts the
  treatment's own spread: a treatment whose replicate wells disagree is measured in a wider covariance.

Three wells are too few to estimate a treatment's covariance on its own, but enough to add to the controls': the
average is positive definite whenever the controls' covariance is, so `"mp_value"` needs only two wells per
treatment. `"hellinger"` compares the two covariances themselves and needs at least twice as many rows as features
on each side, so it does not apply here; the [cell and guide comparison](../genetics/hit_calling_methods.ipynb)
covers it.

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

import mantispy as mt

METHODS = {"mahalanobis": {}, "mp_value": {"method": "mp_value"}}


def call(adata):
    """Run both methods on the PCA representation and return each one's calls at q < 0.05."""
    calls = {}
    for name, keywords in METHODS.items():
        mt.tl.hit_calling(adata, use_rep="X_pca", n_permutations=1000, key_added=name, **keywords)
        calls[name] = adata.uns["mantispy"][name].set_index("group")["is_hit"]
    return pd.DataFrame(calls)


def pseudo_rate(adata, size, n_draws=20, seed=0):
    """Share of control-only pseudo-treatments of ``size`` wells that each method calls at p < 0.05."""
    controls = adata[adata.obs["Metadata_Control_Type"].astype(str).to_numpy() == "negcon"].copy()
    rng = np.random.default_rng(seed)
    called = dict.fromkeys(METHODS, 0)
    for draw in range(n_draws):
        picked = np.zeros(controls.n_obs, dtype=bool)
        picked[rng.choice(controls.n_obs, size, replace=False)] = True
        controls.obs["Metadata_Perturbation"] = np.where(picked, "pseudo", "DMSO")
        # The reference is selected through Metadata_Control_Type, so flip it along with Metadata_Control.
        controls.obs["Metadata_Control_Type"] = np.where(picked, "treatment", "negcon")
        controls.obs["Metadata_Control"] = ~picked
        for name, keywords in METHODS.items():
            mt.tl.hit_calling(controls, use_rep="X_pca", n_permutations=1000, seed=draw, key_added=name, **keywords)
            called[name] += int(controls.uns["mantispy"][name].set_index("group").loc["pseudo", "pvalue"] < 0.05)
    return pd.Series({name: count / n_draws for name, count in called.items()}, name=f"rate at p < 0.05, {size} wells")

## BBBC021

The overview's screen and preprocessing: MCF-7 cells, a compound at one concentration is a treatment, and most
treatments have three replicate wells.

In [2]:
bbbc = mt.ds.bbbc021()
mt.pp.normalize(bbbc, method="mad_robustize", by="Metadata_Plate", reference="negcon", keep_raw=True)
mt.pp.feature_select(bbbc, na_cutoff=0.0)
bbbc = mt.pp.subset_features(bbbc)
sc.pp.pca(bbbc, n_comps=30)

treated = bbbc.obs["Metadata_Control_Type"].astype(str).to_numpy() != "negcon"
wells = bbbc.obs.loc[treated, "Metadata_Perturbation"].astype(str).value_counts()
wells.value_counts().rename("treatments").rename_axis("wells per treatment")

<path>:2: UserWarning: 2 of 467 features have no spread in at least one group of 'Metadata_Plate' among the rows selected by reference='negcon' (Cells_AreaShape_EulerNumber, Nuclei_AreaShape_EulerNumber). They cannot be scaled there, so their values in those groups are set to NaN. They are flagged in var['degenerate_scale'], and pp.feature_select drops them by default; to drop them now, adata = adata[:, ~adata.var['degenerate_scale']].copy(). A variance or outlier cut does not catch a feature that varies across a plate but is constant among its control wells; the flag does.
  mt.pp.normalize(bbbc, method="mad_robustize", by="Metadata_Plate", reference="negcon", keep_raw=True)


<path>:96: UserWarning: 1 selected feature(s) hold values more than outlier_cutoff=500 robust standard deviations from their own median (Cytoplasm_Texture_Gabor_CorrActin_3). They can dominate distances and PCA. Add 'drop_outliers' to operations to drop them, or check where the extreme values come from; on single cells, before aggregation, since a median profile hides a few extreme cells.
  func(target, **params, **extra)


wells per treatment
3    96
2     7
Name: treatments, dtype: int64

First the calibration. Control wells relabelled as a treatment of the typical size should be called at about
the nominal rate.

In [3]:
pseudo_rate(bbbc, size=int(wells.median()))

mahalanobis    0.20
mp_value       0.05
Name: rate at p < 0.05, 3 wells, dtype: float64

In [4]:
bbbc_calls = call(bbbc).loc[wells.index]
pd.crosstab(bbbc_calls["mahalanobis"], bbbc_calls["mp_value"])

mp_value,True
mahalanobis,
False,5
True,98


In [5]:
cells = bbbc.obs.groupby("Metadata_Perturbation", observed=True)["Metadata_CellCount"].mean()
only = bbbc_calls.index[bbbc_calls["mp_value"] & ~bbbc_calls["mahalanobis"]]
pd.DataFrame({"wells": wells[only], "cells per well": cells.reindex(only).round()}).assign(
    control_cells_per_well=round(bbbc.obs.loc[~treated, "Metadata_CellCount"].mean())
)

,wells,cells per well,control_cells_per_well
bryostatin@0.3,2,1576.0,878
PD-169316@10.0,2,1102.0,878
PD-169316@3.0,2,1095.0,878
alsterpaullone@3.0,2,990.0,878
alsterpaullone@1.0,2,1302.0,878


With twenty pseudo-treatments the rate moves in steps of 0.05. `"mp_value"` called one of the twenty at p < 0.05
and the default called four. A calibrated test reaches four of twenty by chance less than 2% of the time, so on these
three-well pseudo-treatments the default runs above the nominal rate and `"mp_value"` does not.

The two methods agree on 98 of the 103 treatments, and `"mp_value"` calls the other five as well, so it calls every
treatment. All five have two wells rather than three, which leaves the default calling two of the seven two-well
treatments. None of the five has lost cells: each keeps more cells per well than the controls' 878.

## JUMP-Target-2

{func}`~mantispy.ds.jump_target2` runs one plate map in eleven laboratories, and its compounds were chosen to cover
many targets rather than to be active, so some of them should not be called. There is no list of which. As a
proxy, {func}`~mantispy.tl.map` with `mode="activity"` asks whether a compound's replicates retrieve each other
against the controls on their own plate, and a compound it misses is not known to be inactive.

In [6]:
jump = mt.ds.jump_target2()
mt.pp.normalize(jump, method="mad_robustize", by="Metadata_Plate", reference="negcon")
mt.pp.feature_select(jump, na_cutoff=0.0)
jump = mt.pp.subset_features(jump)
sc.pp.pca(jump, n_comps=30)

treated = jump.obs["Metadata_Control_Type"].astype(str).to_numpy() != "negcon"
wells = jump.obs.loc[treated, "Metadata_Perturbation"].astype(str).value_counts()
print(f"{wells.size} compounds, {int(wells.median())} wells each (median), {int((~treated).sum())} control wells")
pseudo_rate(jump, size=int(wells.median()))

<path>:2: UserWarning: 111 of 3616 features have no spread in at least one group of 'Metadata_Plate' among the rows selected by reference='negcon' (Cells_AreaShape_EulerNumber, Cells_AreaShape_FormFactor, Cells_Correlation_Manders_DNA_AGP, Cells_Correlation_Manders_ER_AGP, Cells_Correlation_Manders_Mito_AGP and 106 more). They cannot be scaled there, so their values in those groups are set to NaN. They are flagged in var['degenerate_scale'], and pp.feature_select drops them by default; to drop them now, adata = adata[:, ~adata.var['degenerate_scale']].copy(). A variance or outlier cut does not catch a feature that varies across a plate but is constant among its control wells; the flag does.
  mt.pp.normalize(jump, method="mad_robustize", by="Metadata_Plate", reference="negcon")


<path>:96: UserWarning: 28 selected feature(s) hold values more than outlier_cutoff=500 robust standard deviations from their own median (Cells_AreaShape_Compactness, Cells_Correlation_Manders_DNA_ER, Cells_Correlation_Manders_ER_DNA, Cells_Intensity_IntegratedIntensityEdge_Mito, Cells_RadialDistribution_FracAtD_mito_tubeness_1of20 and 23 more). They can dominate distances and PCA. Add 'drop_outliers' to operations to drop them, or check where the extreme values come from; on single cells, before aggregation, since a median profile hides a few extreme cells.
  func(target, **params, **extra)


301 compounds, 14 wells each (median), 898 control wells


mahalanobis    0.1
mp_value       0.1
Name: rate at p < 0.05, 14 wells, dtype: float64

In [7]:
jump_calls = call(jump).loc[wells.index]
mt.tl.map(jump, mode="activity", use_rep="X_pca", key_added="activity")
active = jump.uns["mantispy"]["activity"].set_index("Metadata_Perturbation")["qvalue"].reindex(wells.index) < 0.05
pd.DataFrame(
    {
        name: {
            "called": int(called.sum()),
            "called, mAP active": int((called & active).sum()),
            "called, mAP inactive": int((called & ~active).sum()),
            "mAP active, not called": int((~called & active).sum()),
        }
        for name, called in jump_calls.items()
    }
).T

,called,"called, mAP active","called, mAP inactive","mAP active, not called"
mahalanobis,159,68,91,1
mp_value,172,69,103,0


Control wells relabelled as 14-well compounds are called in two of twenty draws by both methods. `"mp_value"` calls
172 of the 301 compounds and the default 159. Of the 69 compounds mAP calls active, `"mp_value"` calls all 69 and the
default 68. Of the 232 that mAP does not call, `"mp_value"` calls 103 and the default 91.

mAP is not ground truth. A compound it misses may still have a small, real effect, so these extra calls are not known
to be false. They do show that `"mp_value"` returns the longer list on this screen.

## Which to use

`"mahalanobis"` stays the default, and `"mp_value"` is opt-in. On BBBC021 it held the nominal rate on three-well
pseudo-treatments where the default did not, and the treatments it adds are the two-well ones. On JUMP-Target-2 it
calls more of the compounds mAP does not call than the default does, and nothing on this page says whether those
extra calls are right.

On single cells the picture changes. In the [cell and guide comparison](../genetics/hit_calling_methods.ipynb),
`"mp_value"` called nine of the twelve olfactory-receptor negatives of a CRISPR screen, where the default called one.
Whichever method you pick, relabel control wells as pseudo-treatments of your group size, as `pseudo_rate` does
above, to see how often it calls a group that has no effect.